In [ ]:
!pip install -q fastapi uvicorn pyngrok python-multipart nest-asyncio tensorflow pillow opencv-python-headless

In [ ]:
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input
from PIL import Image, ImageOps
import cv2
import io
import base64
from pyngrok import ngrok
import uvicorn
import nest_asyncio
from typing import Dict
import threading
import time
import os

In [ ]:
nest_asyncio.apply()

In [ ]:
MODEL_PATH = os.environ.get("MODEL_PATH", "/content/VGG16_Augmented_last_version.keras")
NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "")

In [ ]:
loaded_model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    denom = tf.math.reduce_max(heatmap) + 1e-10
    heatmap = heatmap / denom
    return heatmap.numpy()

In [ ]:
def process_image(image_bytes: bytes) -> Dict:
    input_img = Image.open(io.BytesIO(image_bytes))
    input_img = ImageOps.exif_transpose(input_img).convert("RGB")
    input_arr = np.array(input_img)

    original_resized_img = cv2.resize(input_arr, (224, 224))

    img_for_model = np.expand_dims(original_resized_img, axis=0).astype(np.float32)
    img_for_model = preprocess_input(img_for_model)

    print("DEBUG: input shape:", img_for_model.shape, "dtype:", img_for_model.dtype,
          "min/max:", img_for_model.min(), img_for_model.max())

    predictions = loaded_model.predict(img_for_model, verbose=0)

    probs = tf.nn.softmax(predictions[0]).numpy()
    predicted_class_index = int(np.argmax(probs))
    predicted_class_name = class_names[predicted_class_index]
    confidence = float(probs[predicted_class_index])

    probabilities = { class_names[i]: float(probs[i]) for i in range(len(class_names)) }

    last_conv_layer_name = 'block5_conv3'
    heatmap = make_gradcam_heatmap(img_for_model, loaded_model, last_conv_layer_name, pred_index=predicted_class_index)

    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored_bgr = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)

    original_bgr = cv2.cvtColor(original_resized_img, cv2.COLOR_RGB2BGR).astype(np.float32)

    superimposed_bgr = cv2.addWeighted(original_bgr, 0.6, heatmap_colored_bgr.astype(np.float32), 0.4, 0)
    superimposed_bgr = np.clip(superimposed_bgr, 0, 255).astype(np.uint8)

    _, gradcam_buffer = cv2.imencode('.jpg', superimposed_bgr)
    gradcam_base64 = base64.b64encode(gradcam_buffer).decode('utf-8')

    print("DEBUG: raw preds (first 8):", predictions[0][:8])
    print("DEBUG: probs sum:", float(np.sum(probs)))

    return {
        "success": True,
        "predicted_class": predicted_class_name,
        "confidence": confidence,
        "probabilities": probabilities,
        "gradcam_image": gradcam_base64,
        "message": "Prediction completed successfully"
    }

In [ ]:
app = FastAPI(title="Brain Tumor Classification API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/")
async def root():
    return {"message":"Brain Tumor Classification API","version":"1.0.0","classes":class_names}

@app.get("/health")
async def health():
    return {"status":"healthy","model_loaded": loaded_model is not None}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    try:
        if not file.content_type or not file.content_type.startswith('image/'):
            raise HTTPException(status_code=400, detail="Invalid file type. Upload an image.")
        image_bytes = await file.read()
        if len(image_bytes) == 0:
            raise HTTPException(status_code=400, detail="Empty file received")
        if len(image_bytes) > 10 * 1024 * 1024:
            raise HTTPException(status_code=400, detail="File too large (max 10MB)")
        result = process_image(image_bytes)
        return JSONResponse(content=result)
    except HTTPException:
        raise
    except Exception as e:
        print(" Error during prediction:", str(e))
        raise HTTPException(status_code=500, detail=f"Processing error: {str(e)}")

# Server run helpers
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

def start_server(ngrok_token: str):
    ngrok.set_auth_token(ngrok_token)
    print(" Starting ngrok tunnel...")
    t = ngrok.connect(8000)
    public_url = t.public_url if hasattr(t, "public_url") else str(t)
    print(f"Public URL: {public_url}")
    print(f"Docs: {public_url}/docs")
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("Server stopped")

if NGROK_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("Please set NGROK_TOKEN before starting the server.")
else:
    print("Starting server...")
    start_server(NGROK_TOKEN)